# Combination of Collaborative Filtering, Content-Based Filtering

## 1. Import Libraries

In [2]:
# HYBRID SCORING ENGINE
# Combining Content-Based (Embeddings) + Collaborative (SVD)

import pandas as pd
import numpy as np
import os
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 2. Load datasets

In [3]:

# Loading data and models from previous files.

# Load enriched movies
enriched_path = '../data/processed/enriched_movies.csv'
enriched_movies = pd.read_csv(enriched_path)
print(f"   Loaded {len(enriched_movies)} enriched movies")
enriched_movies.head(2)

   Loaded 9742 enriched movies


,movieId,title,genres,tmdbId,overview,poster_path,release_date,vote_average,tmdb_genres,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,"Led by Woody, Andy's toys live happily in his ...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,1995-11-22,7.971,"['Family', 'Comedy', 'Animation', 'Adventure']","Toy Story (1995) | Adventure, Animation, Child..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,When siblings Judy and Peter discover an encha...,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,1995-12-15,7.244,"['Adventure', 'Fantasy', 'Family']","Jumanji (1995) | Adventure, Children, Fantasy ..."


In [4]:
# Load embeddings
embeddings_path = '../artifacts/content/movie_embeddings.npy'
movie_embeddings = np.load(embeddings_path)
print(f"   Loaded embeddings shape: {movie_embeddings.shape}")

movie_embeddings[0:1]

   Loaded embeddings shape: (9742, 384)


array([[-7.04915216e-03,  2.04434954e-02,  9.08464193e-02,
        -3.24895047e-02,  2.33547809e-03,  8.14618822e-03,
         8.34751278e-02,  1.28875962e-02, -2.16294527e-02,
         2.62678508e-02,  4.08036374e-02,  3.52204591e-02,
        -2.06873100e-02,  6.71482459e-02,  8.95055979e-02,
         4.57374305e-02, -2.39857123e-03, -4.73910943e-02,
         3.34244929e-02, -7.99269602e-02,  6.88293427e-02,
        -9.54285543e-03,  6.34219348e-02, -3.56637277e-02,
        -7.21041933e-02,  4.49441299e-02,  3.57312127e-03,
        -2.27720197e-02, -5.54080643e-02,  2.48359013e-02,
         7.81237185e-02,  1.75761431e-02, -1.13947270e-02,
        -8.31365064e-02, -4.42581400e-02,  3.93004529e-02,
        -1.05825020e-03, -5.65875098e-02,  7.35560581e-02,
        -4.86732870e-02, -6.45170361e-02,  2.53001023e-02,
        -2.54845768e-02, -1.80552322e-02,  7.20620230e-02,
        -4.99248912e-04, -7.29039758e-02, -5.50034828e-02,
         5.50789349e-02,  6.24580719e-02, -4.47921678e-0

In [5]:
# Load SVD model - Collaborative Filtering
model_path = '../artifacts/svd_model.pkl'
with open(model_path, 'rb') as f:
    svd_model = pickle.load(f)

# Load ratings for reference
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')

## 3. Model Training

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
# defining hybrid function.

def get_hybrid_recommendations(user_id, seed_movie_title=None, top_n=10, alpha=0.6):
    """
    Generate hybrid recommendations.
    
    Parameters:
        user_id: int - Target user
        seed_movie_title: str - Optional seed movie for content boost
        top_n: int - Number of recommendations
        alpha: float (0.0 to 1.0) - Weight for content similarity (higher = more content-focused)
    """
    
    # 1. Get collaborative predictions (SVD)
    print("   Computing collaborative predictions...")
    user_rated = ratings[ratings['userId'] == user_id]['movieId'].unique()
    all_movie_ids = enriched_movies['movieId'].unique()
    unrated_movies = [mid for mid in all_movie_ids if mid not in user_rated]
    
    collab_preds = {}
    for movie_id in tqdm(unrated_movies[:5000], desc="SVD Predictions", leave=False):  # Limit for speed on 8GB
        pred = svd_model.predict(uid=user_id, iid=movie_id)
        collab_preds[movie_id] = pred.est
    
    # 2. Content-based similarity (if seed movie provided)
    content_scores = {}
    if seed_movie_title:
        print(f"   Computing content similarity to seed: {seed_movie_title}")
        # Find seed movie index
        seed_matches = enriched_movies[enriched_movies['title'].str.contains(seed_movie_title, case=False, na=False)]
        if len(seed_matches) == 0:
            print(" Seed movie not found. Skipping content boost.")
            seed_idx = None
        else:
            seed_idx = seed_matches.index[0]
            seed_embedding = movie_embeddings[seed_idx]
            
            # Compute similarity to all movies
            similarities = cosine_similarity([seed_embedding], movie_embeddings)[0]
            
            for i, sim in enumerate(similarities):
                movie_id = enriched_movies.iloc[i]['movieId']
                content_scores[movie_id] = float(sim)
    
    # 3. Hybrid scoring
    hybrid_results = []
    
    for movie_id in collab_preds.keys():
        collab_score = collab_preds[movie_id] / 5.0   # Normalize to 0-1
        
        if seed_movie_title and movie_id in content_scores:
            content_score = content_scores[movie_id]
        else:
            content_score = 0.5  # Neutral if no seed
        
        # Weighted hybrid score
        hybrid_score = (alpha * content_score) + ((1 - alpha) * collab_score)
        
        # Get movie title
        title_row = enriched_movies[enriched_movies['movieId'] == movie_id]
        if len(title_row) == 0:
            continue
        title = title_row.iloc[0]['title']
        
        hybrid_results.append({
            'movieId': movie_id,
            'title': title,
            'hybrid_score': round(hybrid_score, 4),
            'content_score': round(content_score, 4) if seed_movie_title else None,
            'collab_pred': round(collab_preds[movie_id], 2)
        })
    
    # Sort by hybrid score
    hybrid_results.sort(key=lambda x: x['hybrid_score'], reverse=True)
    
    return pd.DataFrame(hybrid_results[:top_n])



## 4. Testing

In [7]:

# Test 1: Pure collaborative (no seed movie, alpha=0)
print("\nTest A: Collaborative-heavy (alpha=0.3) for userId=1")
hybrid_df_a = get_hybrid_recommendations(user_id=1, seed_movie_title=None, top_n=8, alpha=0.3)
print(hybrid_df_a[['title', 'hybrid_score', 'collab_pred']])

# Test 2: With seed movie (more content influence)
print("\nTest B: Content-boosted (alpha=0.7) for userId=1 + seed 'Toy Story'")
hybrid_df_b = get_hybrid_recommendations(user_id=1, seed_movie_title="The Lord of The Rings", top_n=8, alpha=0.7)
print(hybrid_df_b[['title', 'hybrid_score', 'content_score', 'collab_pred']])


Test A: Collaborative-heavy (alpha=0.3) for userId=1
   Computing collaborative predictions...


                                               title  hybrid_score  \
0                   Shawshank Redemption, The (1994)          0.85   
1  Dr. Strangelove or: How I Learned to Stop Worr...          0.85   
2                              Godfather, The (1972)          0.85   
3                         Singin' in the Rain (1952)          0.85   
4                                 Rear Window (1954)          0.85   
5                                  Casablanca (1942)          0.85   
6                   Streetcar Named Desire, A (1951)          0.85   
7  Good, the Bad and the Ugly, The (Buono, il bru...          0.85   

   collab_pred  
0          5.0  
1          5.0  
2          5.0  
3          5.0  
4          5.0  
5          5.0  
6          5.0  
7          5.0  

Test B: Content-boosted (alpha=0.7) for userId=1 + seed 'Toy Story'
   Computing collaborative predictions...


   Computing content similarity to seed: The Lord of The Rings
 Seed movie not found. Skipping content boost.


                                               title  hybrid_score  \
0                   Shawshank Redemption, The (1994)          0.65   
1  Dr. Strangelove or: How I Learned to Stop Worr...          0.65   
2                              Godfather, The (1972)          0.65   
3                         Singin' in the Rain (1952)          0.65   
4                                 Rear Window (1954)          0.65   
5                                  Casablanca (1942)          0.65   
6                   Streetcar Named Desire, A (1951)          0.65   
7  Good, the Bad and the Ugly, The (Buono, il bru...          0.65   

   content_score  collab_pred  
0            0.5          5.0  
1            0.5          5.0  
2            0.5          5.0  
3            0.5          5.0  
4            0.5          5.0  
5            0.5          5.0  
6            0.5          5.0  
7            0.5          5.0  


In [18]:
#  SAVE HYBRID COMPONENTS
print("\n4. Saving hybrid configuration...")

hybrid_dir = '../artifacts'
os.makedirs(hybrid_dir, exist_ok=True)

hybrid_config = {
    'default_alpha': 0.6,
    'description': 'Weighted hybrid: alpha * content_similarity + (1-alpha) * normalized_collab_rating'
}

config_path = os.path.join(hybrid_dir, 'hybrid_config.pkl')
with open(config_path, 'wb') as f:
    pickle.dump(hybrid_config, f)

print(f"Hybrid config saved to: {config_path}")



4. Saving hybrid configuration...
Hybrid config saved to: ../artifacts/hybrid_config.pkl


## Key Achievements

- Weighted hybrid scoring implemented
- Tunable alpha parameter:
  - `0.0` = pure collaborative filtering  
  - `1.0` = pure content-based filtering  
- Supports seed movie:
  - Enables similar style recommendations  
- Normalized scores:
  - Ensures fair combination of different models  
